In [4]:
import pandas as pd
from scipy.io import savemat
import matlab.engine

def create_segment(shift):
    n_initial_240s = 22 + shift
    n_150s = 19
    n_final_240s = 96 - n_initial_240s - n_150s
    
    segment = [120] * n_initial_240s + [0] * n_150s + [120] * n_final_240s
    final_segment = segment * 14
    return final_segment

def generate_sequence(length):
    seq = []
    value = 0
    for _ in range(length):
        seq.append(value)
        value += 1/96
        if round(value, 2) >= 14:
            value = 0
    return seq

def save_to_mat(data):
    df = pd.DataFrame(data, columns=['Value'])
    df['Sequence'] = generate_sequence(len(df))

    # Validate the data shape
    if df.shape != (1344, 2):
        raise ValueError(f"Unexpected data shape: {df.shape}. Expected (1344, 2).")
    
    # Create combined 2D list
    combined_data = df[['Sequence', 'Value']].values.tolist()
    
    # Save this combined list as a single variable in the .mat file
    data_dict_tank3 = {'KLa3_Setpoints_BSM2': combined_data}
    data_dict_tank4 = {'KLa4_Setpoints_BSM2': combined_data}
    data_dict_tank5 = {'KLa5_Setpoints_BSM2': combined_data}
    
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/KLa5_Setpoints_BSM2.mat', data_dict_tank5)
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/KLa4_Setpoints_BSM2.mat', data_dict_tank4)
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/KLa3_Setpoints_BSM2.mat', data_dict_tank3)

def run_matlab_model(iteration):
    eng = matlab.engine.start_matlab()

    # Save the 'iteration' value as a MATLAB datafile
    iteration_data = {"iteration": iteration}
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/iteration.mat', iteration_data)

    eng.cd('/Users/aya/github/WWDR/BSM1-BSM2_MATLAB/BSM2_R2019b', nargout=0)
    eng.run('run_TSaeration_DRbsm2.m', nargout=0)
    eng.quit()

# Get the number of iterations
n_segments = int(input("Enter the number of iterations: "))

shift = 0
for i in range(n_segments):
    data = create_segment(shift)
    save_to_mat(data)
    run_matlab_model(i + 1)  # Pass the current iteration count, starting from 1
    shift += 1



Error using :
Double operands interacting with int64 operands must have integer values.

Error in run_TSaeration_DRbsm2 (line 11)
outputtimes=[0:(1/96):days*96]; %Define the simulation time for dynamic influent

Error in run (line 91)
evalin('caller', strcat(script, ';'));



MatlabExecutionError: 
  File /Users/aya/github/WWDR/BSM1-BSM2_MATLAB/BSM2_R2019b/run_TSaeration_DRbsm2.m, line 11, in run_TSaeration_DRbsm2

  File /Applications/MATLAB_R2022b.app/toolbox/matlab/lang/run.m, line 91, in run
Double operands interacting with int64 operands must have integer values.


In [5]:
import pandas as pd
from scipy.io import savemat
import matlab.engine


create_segment functionality needs to be improved
    - specify nominal and DR KLa values
    - formalize where the DR begins. At Index 22? What time is this?
    - Build on this, specify the length of time for the DR. Seemingly the default case is 4.5 hours.

In [118]:
def initial_create_segment(DR_len, kla, shift):
    n_calibration = (245 + shift*14)*96
    initial_segment = [120] * n_calibration
    
    n_ininominal = 22
    n_DR = DR_len * 4 #22 is roughly 5:30AM
    n_fnlnominal = 96 - n_ininominal - n_DR
    
    segment = [120] * n_ininominal + [kla] * n_DR + [120] * n_fnlnominal
    final_segment = segment * 14
    return initial_segment + final_segment

In [142]:
test_segment = initial_create_segment(10,0,25)
print(test_segment[len(test_segment)-96:len(test_segment)])
print(len(test_segment))

[120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120]
58464


In [136]:
def generate_sequence(length):
    seq = []
    value = 0
    for _ in range(length):
        seq.append(value)
        value += 1/96
        if round(value, 2) >= length:
            value = 0
    return seq


In [137]:
testseq = generate_sequence(len(test_segment))

In [138]:
dftest = pd.DataFrame(test_segment, columns=['Val'])
dftest['Seq'] = testseq
# combo_dftest = df[['Seq','Val']].values.tolist()
dftest[23520:]

,Val,Seq
23520,120,245.000000
23521,120,245.010417
23522,120,245.020833
23523,120,245.031250
23524,120,245.041667
...,...,...
27547,120,286.947917
27548,120,286.958333
27549,120,286.968750
27550,120,286.979167


In [ ]:
indices = dftest.index[dftest['Val'] == 0].tolist()
# print(f"Indices where Value is 0: {indices}")
print(((max(indices)/96)-245-28))

13.635416666666686


## Working Code Below   

In [ ]:
import numpy as np
import pandas as pd
from scipy.io import savemat
import matlab.engine

def create_segment(DR_len, kla, shift):
    n_calibration = (245 + shift*14)*96
    initial_segment = [120] * n_calibration
    
    n_ininominal = 32 #22 is roughly 8:0AM
    n_DR = DR_len * 4 
    n_fnlnominal = 96 - n_ininominal - n_DR
    
    segment = [120] * n_ininominal + [kla] * n_DR + [120] * n_fnlnominal
    final_segment = segment * 14
    return initial_segment + final_segment

def generate_sequence(length):
    seq = []
    value = 0
    for _ in range(length):
        seq.append(value)
        value += 1/96
        if round(value, 2) >= length:
            value = 0
    return seq

def save_to_mat(data):
    df = pd.DataFrame(data, columns=['Value'])
    df['Sequence'] = generate_sequence(len(df))
 
    # Create combined 2D list
    combined_data = df[['Sequence', 'Value']].values.tolist()
    
    # Save this combined list as a single variable in the .mat file
    data_dict_tank3 = {'KLa3_Setpoints_BSM2': combined_data}
    data_dict_tank4 = {'KLa4_Setpoints_BSM2': combined_data}
    data_dict_tank5 = {'KLa5_Setpoints_BSM2': combined_data}
    
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/KLa5_Setpoints_BSM2.mat', data_dict_tank5)
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/KLa4_Setpoints_BSM2.mat', data_dict_tank4)
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/KLa3_Setpoints_BSM2.mat', data_dict_tank3)

def run_matlab_model(iteration):
    eng = matlab.engine.start_matlab()

    # Save the 'iteration' value as a MATLAB datafile
    iteration_data = {"iteration": np.array([iteration], dtype=np.float64)}
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/iteration.mat', iteration_data)

    eng.cd('/Users/aya/github/WWDR/BSM1-BSM2_MATLAB/BSM2_R2019b', nargout=0)
    eng.run('run_TSaeration_DRbsm2.m', nargout=0)
    eng.quit()

# Get the number of iterations
n_segments = int(input("Enter the number of iterations: "))

shift = 0
for i in range(n_segments):
    data = create_segment(5,0,shift)
    save_to_mat(data)
    run_matlab_model(i + 1)  # Pass the current iteration count, starting from 1
    shift += 1



Current iteration value is: 1
 
Running BSM2 to steady state! Solver = ode15s and Simulink model = benchmarkss
**************************************************************************
 
Steady state achieved. Initializing all state variables to steady state values.
 
Simulating BSM2 with dynamic influent (i) in open loop (Tempmodel = 1)! Solver = ode45 and Simulink model = benchmark
*****************************************************************************************************************
 
Start time for simulation (hour:min:sec) = 22  35  29
the MATLAB function has been cancelled


Error using run_TSaeration_DRbsm2
Error in 'DR_bsm2_ol/ActivatedSludge/Hyd_delay/Hyd_delay' while executing C MEX
S-function 'hyddelayv3_bsm2', (mdlOutputs), at time 7.9630161571296156.

Error in run (line 91)
evalin('caller', strcat(script, ';')); - Show complete stack trace
Caused by:
    Error using run_TSaeration_DRbsm2
    Program interruption (Ctrl-C) has been detected. - Show complete stack trace



# Scratch work ahead

In [ ]:
import numpy as np
import pandas as pd
from scipy.io import savemat
import matlab.engine
import os

def create_kla_sequence(days, DR_len, kla_value):
    n_days = int(days)
    n_steps = n_days * 96
    n_ininominal = 22
    n_DR = DR_len * 4
    n_fnlnominal = 96 - n_ininominal - n_DR
    day_pattern = [120] * n_ininominal + [kla_value] * n_DR + [120] * n_fnlnominal
    full_sequence = day_pattern * n_days
    return full_sequence

def generate_sequence(length):
    return [i / 96 for i in range(length)]

def save_kla_to_mat(kla_sequence):
    df = pd.DataFrame(kla_sequence, columns=['Value'])
    df['Sequence'] = generate_sequence(len(df))
    combined = df[['Sequence', 'Value']].values.tolist()
    for tank in [3, 4, 5]:
        data_dict = {f'KLa{tank}_Setpoints_BSM2': combined}
        savemat(f'BSM1-BSM2_MATLAB/BSM2_R2019b/KLa{tank}_Setpoints_BSM2.mat', data_dict)

def run_bsm2_model(state_in, state_out, output_file, t0, duration):
    eng = matlab.engine.start_matlab()
    eng.cd('/Users/aya/github/WWDR/BSM1-BSM2_MATLAB/BSM2_R2019b', nargout=0)
    eng.workspace['state_in'] = state_in
    eng.workspace['state_out'] = state_out
    eng.workspace['output_file'] = output_file
    eng.workspace['t0_days'] = float(t0)
    eng.workspace['duration_days'] = float(duration)
    eng.eval("run_bsm2_iteration_workflow(state_in, state_out, output_file, t0_days, duration_days)", nargout=0)
    eng.quit()

def main():
    n_iterations = int(input("Enter the number of iterations: "))
    base_dir = '/Users/aya/github/WWDR/BSM1-BSM2_MATLAB'
    os.makedirs(f'{base_dir}/SavedStates', exist_ok=True)
    os.makedirs(f'{base_dir}/SimResults', exist_ok=True)

    # Step 0: Run burn-in to generate state_iter0.mat
    kla_sequence = create_kla_sequence(245, 0, 120)
    save_kla_to_mat(kla_sequence)
    run_bsm2_model('', f'{base_dir}/SavedStates/state_iter0.mat', '', 0, 245)

    for i in range(n_iterations):
        t_start = 245 + i * 14

        # Step 1: Experimental run
        kla_sequence = create_kla_sequence(14, 10, 0)
        save_kla_to_mat(kla_sequence)
        run_bsm2_model(
            f'{base_dir}/SavedStates/state_iter{i}.mat',
            '',
            f'{base_dir}/SimResults/iter{i+1}_output.mat',
            t_start, 14)

        # Step 2: Nominal recovery
        kla_sequence = create_kla_sequence(14, 0, 120)
        save_kla_to_mat(kla_sequence)
        run_bsm2_model(
            f'{base_dir}/SavedStates/state_iter{i}.mat',
            f'{base_dir}/SavedStates/state_iter{i+1}.mat',
            '',
            t_start, 14)

if __name__ == '__main__':
    main()



=== Starting BSM2 Iteration Workflow ===
t0_days = 0.00, duration_days = 245.00

Running BSM2 steady-state initialization using DR_bsm2_ss...
Steady state initialization complete.
Saving initial steady state to: /Users/aya/github/WWDR/BSM1-BSM2_MATLAB/SavedStates/state_iter0.mat

=== Starting BSM2 Iteration Workflow ===
t0_days = 245.00, duration_days = 14.00

Running BSM2 steady-state initialization using DR_bsm2_ss...
Steady state initialization complete.

Running dynamic open-loop BSM2 simulation from timestep 23520.00 to 24864.00...


Error using run_bsm2_iteration_workflow
Output times for the block diagram 'DR_bsm2_ol' must be within the specified
simulation start (23520.0) and stop (24864.0) times.



MatlabExecutionError: Output times for the block diagram 'DR_bsm2_ol' must be within the specified simulation start (23520.0) and stop (24864.0) times.


In [ ]:
import numpy as np
import pandas as pd
from scipy.io import savemat
import matlab.engine
import os

def create_kla_sequence(days, DR_len, kla_value):
    n_days = int(days)
    n_steps = n_days * 96
    n_ininominal = 22
    n_DR = DR_len * 4
    n_fnlnominal = 96 - n_ininominal - n_DR
    day_pattern = [120] * n_ininominal + [kla_value] * n_DR + [120] * n_fnlnominal
    full_sequence = day_pattern * n_days
    return full_sequence

def generate_sequence(length):
    return [i / 96 for i in range(length)]

def save_kla_to_mat(kla_sequence):
    df = pd.DataFrame(kla_sequence, columns=['Value'])
    df['Sequence'] = generate_sequence(len(df))
    combined = df[['Sequence', 'Value']].values.tolist()
    for tank in [3, 4, 5]:
        data_dict = {f'KLa{tank}_Setpoints_BSM2': combined}
        savemat(f'BSM1-BSM2_MATLAB/BSM2_R2019b/KLa{tank}_Setpoints_BSM2.mat', data_dict)

def run_bsm2_model(t0_days, duration, count):
    eng = matlab.engine.start_matlab()
    eng.cd('/Users/aya/github/WWDR/BSM1-BSM2_MATLAB/BSM2_R2019b', nargout=0)

    # Convert Python None to MATLAB empty string
    eng.workspace['t0_days'] = float(t0)
    eng.workspace['duration_days'] = float(duration)
    eng.workspace['iteration'] = float(count)

    eng.eval("run_bsm2_simple_iteration(t0_days, Rt, iteration)", nargout=0)
    eng.quit()

def main():
    n_iterations = int(input("Enter the number of iterations: "))
    base_dir = '/Users/aya/github/WWDR/BSM1-BSM2_MATLAB'
    os.makedirs(f'{base_dir}/SavedStates', exist_ok=True)
    os.makedirs(f'{base_dir}/SimResults', exist_ok=True)

    # Step 0: Run burn-in to generate initial states_bsm2.mat
    kla_sequence = create_kla_sequence(245, 0, 120)
    save_kla_to_mat(kla_sequence)
    run_bsm2_model(0, 245)

    for i in range(n_iterations):
        t_start = 245 + i * 14

        # Step 1: Experimental run from saved states_bsm2.mat
        kla_sequence = create_kla_sequence(14, 10, 0)
        save_kla_to_mat(kla_sequence)
        run_bsm2_model(t_start, 14)

        # Step 2: Recovery run again from states_bsm2.mat
        kla_sequence = create_kla_sequence(14, 0, 120)
        save_kla_to_mat(kla_sequence)
        run_bsm2_model(t_start, 14)

if __name__ == '__main__':
    main()


TypeError: run_bsm2_model() missing 1 required positional argument: 'count'